In [ ]:
# NOTEBOOK NAME
# FeatureStatTimeChunks.ipynb
# NOTEBOOK NAME

# OPENING IMPORTS
import numpy as np
from matplotlib import pyplot as plt
import xarray as xr

import math
import calendar

# SPECIAL METHOD TO IMPORT CUSTOM FUNCTIONS LOCAL DIRECTORY
import sys
sys.path.append('/home/563/sg3241/Notebooks/PhD/CustomFunctions/')
from CustomFunctions1 import *
from RadarPlotsCustomFunctions import *

from pyproj import Geod

In [ ]:
# calculating distances using lats and lons

def latlon_to_xy_dist(lat1, lon1, lat2, lon2):
    """
    Calculate the X (East-West) and Y (North-South) distance
    between two lat/lon points in kilometres.

    Parameters
    ----------
    lat1 : float  - Latitude  of Point 1 (degrees)
    lon1 : float  - Longitude of Point 1 (degrees)
    lat2 : float  - Latitude  of Point 2 (degrees)
    lon2 : float  - Longitude of Point 2 (degrees)

    Returns
    -------
    dist_x_km : float - East-West   distance in m (positive = East)
    dist_y_km : float - North-South distance in m (positive = North)
    """

    geod = Geod(ellps='WGS84')

    az_fwd, _, dist_m = geod.inv(lon1, lat1, lon2, lat2)

    az_rad = np.deg2rad(az_fwd)

    dist_x_m = (dist_m * np.sin(az_rad)) 
    dist_y_m = (dist_m * np.cos(az_rad)) 

    return dist_x_m, dist_y_m

In [ ]:

FileDateStr = '20240202'
RadarIDno = 22
QualityControlOption = 2

FeatureStoragePath = (f'/scratch/v46/sg3241/tmp/NetCDFs/PyFLEXTRKR/Stats/{RadarIDno}/{FileDateStr}/QC{QualityControlOption}/V5/trackstats_{FileDateStr}.000000_{FileDateStr}.235500.nc')

FeatureXR = xr.open_dataset(FeatureStoragePath)

FeatureXR = AddPropagationVars(FeatureXR)

FeatureXR = AddFrameTimeVars(FeatureXR)


In [ ]:
FeatureXR = AddFrameTimeVars(FeatureXR)

In [ ]:
FeatureXR

In [ ]:
track_i = 0
time_i = 0

StartTime = FeatureXR['base_time'][track_i][time_i].values
EndTime   = FeatureXR['base_time'][track_i][time_i+1].values
# Calculate the change in time in seconds
Tdiff = (EndTime - StartTime) / np.timedelta64(1, 's')

StartLat  = FeatureXR['meanlat'][track_i][time_i].values
StartLon  = FeatureXR['meanlon'][track_i][time_i].values

EndLat    = FeatureXR['meanlat'][track_i][time_i+1].values
EndLon    = FeatureXR['meanlon'][track_i][time_i+1].values

# calculate distance change in the feature centre coordinates [m]
Xtravel, Ytravel = latlon_to_xy_dist(StartLat, StartLon, EndLat, EndLon)

# calculate the component propogation speeds in [m/s]
Uspeed = Xtravel / Tdiff
Vspeed = Ytravel / Tdiff 

# calculate the propogation vector magnitude and direction
PropSpeed     = (Uspeed**2 + Vspeed**2) ** (0.5)
PropDirection = 180 + np.rad2deg(np.arctan2(Uspeed,Vspeed)) # note that we are using smart arctan (arctan2)
# Swapping the arguments to arctan2 to be (x,y) is the trick that makes 0° point North and angles increase clockwise
# add 180 to reverse the direction since a positive, positive wind is actually a southwest wind in the third quadrant

In [ ]:
# CHAD AIDED VECTORISED "LOOP"
# CALCULATES PROPOGATION COMPONENTS AND SPEED AND DIRECTION FOR ALL FEATURES AT EACH TIME STEP DIFFERENCE

# --- Load full arrays into memory once (much faster than indexing xarray in a loop) ---
base_time = FeatureXR['base_time'].values   # (tracks, times)
meanlat   = FeatureXR['meanlat'].values     # (tracks, times)
meanlon   = FeatureXR['meanlon'].values     # (tracks, times)

# --- Compute time differences in seconds (all at once) ---
# Shift along time axis: [t] - [t-1]
Tdiff = (base_time[:, 1:] - base_time[:, :-1]) / np.timedelta64(1, 's')
# Shape: (tracks, times-1)

# --- Grab start/end lat/lon slices ---
StartLat = meanlat[:, :-1]   # (tracks, times-1)
StartLon = meanlon[:, :-1]
EndLat   = meanlat[:, 1:]
EndLon   = meanlon[:, 1:]

# --- Vectorised latlon_to_xy_dist using pyproj ---
from pyproj import Geod
geod = Geod(ellps='WGS84')

# Flatten to 1D for pyproj (it handles arrays natively)
flat_lon1 = StartLon.ravel()
flat_lat1 = StartLat.ravel()
flat_lon2 = EndLon.ravel()
flat_lat2 = EndLat.ravel()

# Mask where ANY of the four values are NaN or Tdiff is 0
valid_mask = (
    ~np.isnan(flat_lat1) &
    ~np.isnan(flat_lon1) &
    ~np.isnan(flat_lat2) &
    ~np.isnan(flat_lon2) &
    (Tdiff.ravel() != 0)
)

# --- Pre-allocate flat output arrays ---
n_flat = flat_lon1.shape[0]
Xtravel_flat = np.full(n_flat, np.nan)
Ytravel_flat = np.full(n_flat, np.nan)

# --- Run pyproj only on valid points ---
az_fwd, _, dist_m = geod.inv(
    flat_lon1[valid_mask],
    flat_lat1[valid_mask],
    flat_lon2[valid_mask],
    flat_lat2[valid_mask]
)

az_rad = np.deg2rad(az_fwd)
Xtravel_flat[valid_mask] = (dist_m * np.sin(az_rad)) 
Ytravel_flat[valid_mask] = (dist_m * np.cos(az_rad)) 

# --- Reshape back to (tracks, times-1) ---
Xtravel = Xtravel_flat.reshape(n_tracks, n_times - 1)
Ytravel = Ytravel_flat.reshape(n_tracks, n_times - 1)

# --- Compute speeds ---
Uspeed = Xtravel / Tdiff   # NaN propagates naturally where invalid
Vspeed = Ytravel / Tdiff

PropSpeed     = (Uspeed**2 + Vspeed**2) ** 0.5
PropDirection = 180 + np.rad2deg(np.arctan2(Uspeed, Vspeed))

# --- Pad a column of NaNs at the front (time_i=0 has no previous step) ---
nan_col = np.full((n_tracks, 1), np.nan)

Uspeed_all    = np.hstack([nan_col, Uspeed])
Vspeed_all    = np.hstack([nan_col, Vspeed])
PropSpeed_all = np.hstack([nan_col, PropSpeed])
PropDir_all   = np.hstack([nan_col, PropDirection])

# --- Add new variables to FeatureXR ---
FeatureXR = FeatureXR.assign(

    u_prop_speed = xr.DataArray(
        Uspeed_all,
        dims   = ['tracks', 'times'],
        attrs  = {
            'long_name' : 'Eastward Propagation Speed',
            'units'     : 'm s-1',
            'comments'  : 'East-West component of feature propagation speed. '
                          'Positive values indicate eastward motion.',
        }
    ),

    v_prop_speed = xr.DataArray(
        Vspeed_all,
        dims   = ['tracks', 'times'],
        attrs  = {
            'long_name' : 'Northward Propagation Speed',
            'units'     : 'm s-1',
            'comments'  : 'North-South component of feature propagation speed. '
                          'Positive values indicate northward motion.',
        }
    ),

    total_prop_speed = xr.DataArray(
        PropSpeed_all,
        dims   = ['tracks', 'times'],
        attrs  = {
            'long_name' : 'Total Propagation Speed',
            'units'     : 'm s-1',
            'comments'  : 'Magnitude of the feature propagation speed vector. '
                          'Computed as sqrt(u_prop_speed^2 + v_prop_speed^2).',
        }
    ),

    prop_direction = xr.DataArray(
        PropDir_all,
        dims   = ['tracks', 'times'],
        attrs  = {
            'long_name' : 'Propagation Direction',
            'units'     : 'degrees',
            'comments'  : 'Direction of feature propagation as an azimuthal bearing. '
                          'Measured clockwise from North (0-360°). '
                          '0° = North, 90° = East, 180° = South, 270° = West.',
        }
    ),

)

In [ ]:
FeatureXR

In [ ]:
# CHAD FUNCTION
# CREATES ARRAYS FROM SPECIFIC VARIABLES IN FRAMETIMES
# USED TO LATER FIND MEANS AND SUCH

def TimeChunkArray(variable: str, time_range: str, min_lifetime: int, ds) -> np.ndarray:
    """
    Extracts a numpy array from an xarray Dataset for a given variable,
    time range, and minimum track lifetime.

    Parameters
    ----------
    variable : str
        Name of the variable in the dataset (e.g., 'area_frametimes').
    time_range : str
        Time range in 'HH:MM-HH:MM' format (e.g., '14:00-16:00').
    min_lifetime : int
        Minimum lifetime in minutes a track must last to be included.
        Set to 0 to include all tracks.
    ds : xarray.Dataset
        The xarray Dataset to extract from.

    Returns
    -------
    np.ndarray
        Array of shape (n_tracks, n_frametimes) for the specified time window,
        filtered by minimum lifetime.
    """

    # --- Parse the time range string ---
    start_str, end_str = time_range.split('-')

    start_h, start_m = map(int, start_str.split(':'))
    end_h,   end_m   = map(int,   end_str.split(':'))

    start_minutes = start_h * 60 + start_m
    end_minutes   = end_h   * 60 + end_m

    # --- Convert minutes to FrameTime indices ---
    start_idx = start_minutes // 5
    end_idx   = (end_minutes  // 5) - 1

    # --- Compute minimum number of frames required ---
    # ceil(min_lifetime / 5) + 1 works universally:
    # 0 min -> 1 frame (all tracks qualify, since every track has >= 1 frame)
    # 5 min -> 2 frames, 6 min -> 3 frames, 10 min -> 3 frames, etc.
    min_frames = math.ceil(min_lifetime / 5) + 1

    # --- Filter tracks by lifetime ---
    track_mask = ds['track_duration'].values >= min_frames

    # --- Extract variable, filter tracks, return ---
    data = ds[variable].isel(FrameTimes=slice(start_idx, end_idx + 1))

    return data.isel(tracks=track_mask).values


In [ ]:
# Execution Block
# Creating Arrays of Time Chunk Means

QualityControlOption = 2

RadarIDno = 22

PlotYear  = '2024'
PlotMonth = '02'

# Choose how large of chunks you want statss for (should be a factor of 1440 minutes in a day)
chunk_size = 30  # minutes
n_chunks = (24 * 60) // chunk_size  # e.g. 48 for 30-min chunks

# Generate all days in the month
num_days = calendar.monthrange(int(PlotYear), int(PlotMonth))[1]
all_days = [f"{PlotYear}-{PlotMonth}-{d:02d}" for d in range(1, num_days + 1)]

NumDays = len(all_days)
MeanArea          = np.full((NumDays, n_chunks), np.nan)
MeanDepth         = np.full((NumDays, n_chunks), np.nan)
MeanIntensity     = np.full((NumDays, n_chunks), np.nan)
MeanPropSpeedU    = np.full((NumDays, n_chunks), np.nan)
MeanPropSpeedV    = np.full((NumDays, n_chunks), np.nan)
MeanPropSpeed     = np.full((NumDays, n_chunks), np.nan)
MeanPropDirection = np.full((NumDays, n_chunks), np.nan)

for day_idx, PlotDate in enumerate(all_days):

    # Reconstruct date string
    YearStr     = PlotDate[0:4]
    MonthStr    = PlotDate[5:7]
    DayStr      = PlotDate[8:10]
    FileDateStr = YearStr + MonthStr + DayStr

    print(f'Working on {FileDateStr}')

    FeatureStoragePath = (f'/scratch/v46/sg3241/tmp/NetCDFs/PyFLEXTRKR/Stats/{RadarIDno}/{FileDateStr}/QC{QualityControlOption}/V5/trackstats_{FileDateStr}.000000_{FileDateStr}.235500.nc')

    # Skip days where the file doesn't exist
    try:
        FeatureXR = xr.open_dataset(FeatureStoragePath)
    except FileNotFoundError:
        print(f"File not found for {PlotDate}, skipping.")
        continue

    # add propogation variables
    FeatureXR = AddPropagationVars(FeatureXR)
        
    # add variables that rearrange times by time of day rather than time in a given feature's life
    FeatureXR = AddFrameTimeVars(FeatureXR)

    # Inner loop over time chunks
    for i in range(n_chunks):
        start_minutes = i * chunk_size
        end_minutes   = start_minutes + chunk_size

        start_str  = f"{start_minutes // 60:02d}:{start_minutes % 60:02d}"
        end_str    = f"{end_minutes   // 60:02d}:{end_minutes   % 60:02d}"
        time_range = f"{start_str}-{end_str}"

        MeanArea[day_idx, i]          = np.nanmean(TimeChunkArray('area_frametimes',             time_range, 6, FeatureXR))
        MeanDepth[day_idx, i]         = np.nanmean(TimeChunkArray('maxETH_20dbz_frametimes',     time_range, 6, FeatureXR))
        MeanIntensity[day_idx, i]     = np.nanmean(TimeChunkArray('max_dbz_frametimes',          time_range, 6, FeatureXR))
        MeanPropSpeedU[day_idx, i]    = np.nanmean(TimeChunkArray('u_prop_speed_frametimes', time_range, 6, FeatureXR))
        MeanPropSpeedV[day_idx, i]    = np.nanmean(TimeChunkArray('v_prop_speed_frametimes', time_range, 6, FeatureXR))
        MeanPropSpeed[day_idx, i]     = np.nanmean(TimeChunkArray('total_prop_speed_frametimes', time_range, 6, FeatureXR))
        MeanPropDirection[day_idx, i] = np.nanmean(TimeChunkArray('prop_direction_frametimes',   time_range, 6, FeatureXR))

    FeatureXR.close()

In [ ]:
FeatureXR


In [ ]:
# CHAD PLOT
# DIURNAL CYCLE PERCENTILE DISTRIBUTIONS

PlotYear  = '2024'
PlotMonth = '02'

LineColour = [1, 0, 0]  # red
MinSampleSize = 5  # minimum number of valid days needed to plot a chunk

variables = {
    'area_frametimes'         : 'Mean Area',
    'max_dbz_frametimes'      : 'Mean Intensity',
    'maxETH_20dbz_frametimes' : 'Mean Depth',
}

# --- Tick positions for x-axis ---
ticks_per_hour = 60 // chunk_size
tick_positions = np.arange(0, n_chunks, ticks_per_hour)
tick_labels    = [
    f"{(i * chunk_size) // 60:02d}:{(i * chunk_size) % 60:02d}"
    for i in tick_positions
]

# --- Plotting helper (same style as reference code) ---
def plot_percentiles_tod(ax, x, pct_10, pct_25, pct_50, pct_75, pct_90, colour):
    ax.fill_between(x, pct_10, pct_90,
                    color=colour, alpha=0.14, linewidth=0)
    ax.fill_between(x, pct_25, pct_75,
                    color=colour, alpha=0.20, linewidth=0)
    ax.plot(x, pct_10, color=colour, linewidth=0.5, alpha=0.6, linestyle='--')
    ax.plot(x, pct_90, color=colour, linewidth=0.5, alpha=0.6, linestyle='--')
    ax.plot(x, pct_25, color=colour, linewidth=1.0, alpha=0.6, linestyle='-')
    ax.plot(x, pct_75, color=colour, linewidth=1.0, alpha=0.6, linestyle='-')
    ax.plot(x, pct_50, color=colour, linewidth=2.5, alpha=1.0, linestyle='-')

# --- Loop over each variable and produce one figure each ---
for var, label in variables.items():

    # MeanArea / MeanDepth / MeanIntensity are (NumDays x n_chunks)
    # Map label back to the correct 2D array
    if label == 'Mean Area':
        data_2d = MeanArea
    elif label == 'Mean Intensity':
        data_2d = MeanIntensity
    elif label == 'Mean Depth':
        data_2d = MeanDepth

    # --- Compute percentiles across days for each chunk ---
    x       = np.arange(n_chunks)
    pct_10  = np.full(n_chunks, np.nan)
    pct_25  = np.full(n_chunks, np.nan)
    pct_50  = np.full(n_chunks, np.nan)
    pct_75  = np.full(n_chunks, np.nan)
    pct_90  = np.full(n_chunks, np.nan)
    sample_counts = np.zeros(n_chunks, dtype=int)

    for i in range(n_chunks):
        col = data_2d[:, i]                  # all days for this chunk
        valid = col[~np.isnan(col)]          # drop NaN (missing days)
        sample_counts[i] = len(valid)

        if len(valid) < MinSampleSize:
            continue                          # leave as NaN if too few days

        pct_10[i] = np.percentile(valid, 10)
        pct_25[i] = np.percentile(valid, 25)
        pct_50[i] = np.percentile(valid, 50)
        pct_75[i] = np.percentile(valid, 75)
        pct_90[i] = np.percentile(valid, 90)

    # --- Set up figure with histogram panel below ---
    fig = plt.figure(figsize=(14, 6), facecolor='black')
    gs  = fig.add_gridspec(2, 1, height_ratios=[4, 1], hspace=0.05)

    ax_lines = fig.add_subplot(gs[0])
    ax_hist  = fig.add_subplot(gs[1], sharex=ax_lines)

    ax_lines.set_facecolor('black')
    ax_hist.set_facecolor('black')

    # --- Percentile lines and fills ---
    plot_percentiles_tod(
        ax_lines, x,
        pct_10, pct_25, pct_50, pct_75, pct_90,
        LineColour
    )

    # --- Line plot formatting ---
    ax_lines.set_xticks(tick_positions)
    ax_lines.set_xlim(0, n_chunks - 1)
    ax_lines.set_ylabel(label, color='white', fontsize=13)
    ax_lines.set_title(
        f'{label} by Time of Day — {PlotYear}-{PlotMonth} (Percentiles Across Days)',
        color='white', fontsize=14
    )
    ax_lines.tick_params(colors='white', which='both', labelsize=10)
    ax_lines.grid(which='major', color='white', linewidth=0.8, linestyle='-', alpha=0.6)
    ax_lines.grid(which='minor', color='white', linewidth=0.3, linestyle='-', alpha=0.3)
    ax_lines.minorticks_on()
    plt.setp(ax_lines.get_xticklabels(), visible=False)

    for spine in ax_lines.spines.values():
        spine.set_edgecolor('white')

    # --- Legend ---
    from matplotlib.lines import Line2D
    legend_elements = [
        Line2D([0], [0], color=LineColour, linewidth=2.5,
               label=f'{PlotYear}-{PlotMonth} Median'),
        Line2D([0], [0], color=LineColour, linewidth=1.0, alpha=0.6,
               label='25th / 75th Percentile'),
        Line2D([0], [0], color=LineColour, linewidth=0.5, alpha=0.6,
               linestyle='--', label='10th / 90th Percentile'),
    ]
    ax_lines.legend(
        handles=legend_elements,
        facecolor='black', edgecolor='white',
        labelcolor='white', fontsize=11
    )

    # --- Bottom histogram: number of valid days per chunk ---
    ax_hist.bar(
        x, sample_counts,
        width=0.8, color=LineColour, edgecolor='white', alpha=0.6
    )
    ax_hist.set_xlim(0, n_chunks - 1)
    ax_hist.set_xticks(tick_positions)
    ax_hist.set_xticklabels(tick_labels, rotation=45, ha='right',
                             color='white', fontsize=8)
    ax_hist.set_xlabel('Time of Day (UTC)', color='white', fontsize=13)
    ax_hist.set_ylabel('Days\nwith Data', color='white', fontsize=10)
    ax_hist.tick_params(colors='white', which='both', labelsize=9)
    ax_hist.set_ylim(0, NumDays)

    for spine in ax_hist.spines.values():
        spine.set_edgecolor('white')

    # --- Align panel widths ---
    plt.tight_layout()
    plt.draw()

    ax_lines_pos = ax_lines.get_position()
    ax_hist.set_position([
        ax_lines_pos.x0,
        ax_hist.get_position().y0,
        ax_lines_pos.width,
        ax_hist.get_position().height
    ])

    plt.show()


In [ ]:
FeatureXR

In [ ]:

chunk_size = 30  # minutes
n_chunks = (24 * 60) // chunk_size  # e.g. 48 for 30-min chunks

MeanArea = np.full(n_chunks, np.nan)
MeanDepth = np.full(n_chunks, np.nan)
MeanIntensity = np.full(n_chunks, np.nan)
MeanDirection = np.full(n_chunks, np.nan)
MeanSpeed = np.full(n_chunks, np.nan)

for i in range(n_chunks):
    start_minutes = i * chunk_size
    end_minutes   = start_minutes + chunk_size

    start_str = f"{start_minutes // 60:02d}:{start_minutes % 60:02d}"
    end_str   = f"{end_minutes   // 60:02d}:{end_minutes   % 60:02d}"
    time_range = f"{start_str}-{end_str}"

    MeanArea[i]      = np.nanmean( TimeChunkArray('area_frametimes', time_range, 6, FeatureXR) )
    MeanDepth[i]     = np.nanmean( TimeChunkArray('maxETH_20dbz_frametimes', time_range, 6, FeatureXR) )
    MeanIntensity[i] = np.nanmean( TimeChunkArray('max_dbz_frametimes', time_range, 6, FeatureXR) )
    MeanDirection[i] = np.nanmean( TimeChunkArray('prop_direction_frametimes', time_range, 6, FeatureXR) )
    MeanSpeed[i]     = np.nanmean( TimeChunkArray('total_prop_speed_frametimes', time_range, 6, FeatureXR) )



In [ ]:


# CHAD PLOT
# MEAN FEATURE CHARACTERISTICS OVER TIMES OF DAY

variables = {
    'prop_direction_frametimes'      : 'Mean Direction',
    'total_prop_speed_frametimes' : 'Mean Speed',
}

results = {label: np.full(n_chunks, np.nan) for label in variables.values()}

# --- Compute means ---
for var, label in variables.items():
    for i in range(n_chunks):
        start_minutes = i * chunk_size
        end_minutes   = start_minutes + chunk_size

        start_str  = f"{start_minutes // 60:02d}:{start_minutes % 60:02d}"
        end_str    = f"{end_minutes   // 60:02d}:{end_minutes   % 60:02d}"
        time_range = f"{start_str}-{end_str}"

        SampleArray      = TimeChunkArray(var, time_range, 6, FeatureXR)
        results[label][i] = np.nanmean(SampleArray)

# --- Plot ---
ticks_per_hour = 60 // chunk_size
tick_positions = np.arange(0, n_chunks, ticks_per_hour)
tick_labels    = [f"{(i * chunk_size) // 60:02d}:{(i * chunk_size) % 60:02d}" for i in tick_positions]

for label, data in results.items():
    fig, ax = plt.subplots(figsize=(14, 5), facecolor='black')
    ax.set_facecolor('black')

    ax.plot(np.arange(n_chunks), data, color='white', linewidth=1)

    ax.set_xticks(tick_positions)
    ax.set_xticklabels(tick_labels, rotation=45, ha='right', color='white', fontsize=8)
    ax.set_xlim(0, n_chunks - 1)

    ax.yaxis.label.set_color('white')
    ax.xaxis.label.set_color('white')
    ax.tick_params(axis='y', colors='white')
    ax.spines['bottom'].set_color('white')
    ax.spines['left'].set_color('white')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

    ax.set_xlabel('Time of Day (UTC)', color='white')
    ax.set_ylabel(label, color='white')
    ax.set_title(f'{label} by Time of Day', color='white')

    plt.tight_layout()
    plt.show()



In [ ]:
FeatureXR['base_time'].values

In [ ]:
# CHAD CREATED FEATURE BIRTH LOCATIONS MAP

PlotType = 'density' # 'scatter' for time-of-day coloured scatter plot or 'density' for kernel density contours

# ── extract birth locations and times ─────────────────────────────────────────
time_coords  = pd.to_datetime(FeatureXR['time'].values)
born_arr     = FeatureXR['base_time'].values          # (n_times, n_features)
centre_lat   = FeatureXR['meanlat'].values    # (n_times, n_features)
centre_lon   = FeatureXR['meanlon'].values    # (n_times, n_features)

birth_lats   = []
birth_lons   = []
birth_mins   = []   # time of day in minutes for colouring

for t_idx, ts in enumerate(time_coords):
    # Find all features born at this timestep
    born_mask = born_arr[t_idx, :] == 1.0
    if not np.any(born_mask):
        continue
    lats = centre_lat[t_idx, born_mask]
    lons = centre_lon[t_idx, born_mask]
    # Time of day in minutes
    minutes = ts.hour * 60 + ts.minute + ts.second / 60.0
    for lat, lon in zip(lats, lons):
        if np.isfinite(lat) and np.isfinite(lon):
            birth_lats.append(lat)
            birth_lons.append(lon)
            birth_mins.append(minutes)

birth_lats = np.array(birth_lats)
birth_lons = np.array(birth_lons)
birth_mins = np.array(birth_mins)

print(f"Plotting {len(birth_lats)} feature birth locations.")

# ── set up figure with topo background ───────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 6),
                       subplot_kw={'projection': ccrs.PlateCarree()})

lon_min = float(np.nanmin(centre_lon[np.isfinite(centre_lon)]))
lon_max = float(np.nanmax(centre_lon[np.isfinite(centre_lon)]))
lat_min = float(np.nanmin(centre_lat[np.isfinite(centre_lat)]))
lat_max = float(np.nanmax(centre_lat[np.isfinite(centre_lat)]))

# ── topo background ───────────────────────────────────────────────────────────
try:
    gebco_path = '/home/563/sg3241/QueenslandElevationGEBCO.nc'
    dem_da     = fetch_gebco_local(gebco_path, lon_min, lon_max, lat_min, lat_max)

    if dem_da is not None:
        dem_lon  = dem_da.lon.values
        dem_lat  = dem_da.lat.values
        dem_data = dem_da.values

        if dem_lon.ndim == 1 and dem_lat.ndim == 1:
            dem_lon_2d, dem_lat_2d = np.meshgrid(dem_lon, dem_lat)
        else:
            dem_lon_2d, dem_lat_2d = dem_lon, dem_lat

        if not np.any(np.isfinite(dem_data)):
            raise ValueError('DEM has no finite values in this domain')

        colours_topo = [
            '#dde4e8',
            '#c4dec2',
            '#e4edc9',
            '#f3f0cf',
            '#e9d7bd',
            '#ddc4aa',
            '#cfb194',
            '#b58f6e',
        ]
        bounds_topo = [-1000.0, 0.0, 200.0, 400.0, 600.0, 800.0, 1000.0, 1200.0, 5000.0]

        cmap_elev = ListedColormap(colours_topo)
        norm_topo = BoundaryNorm(bounds_topo, len(colours_topo), clip=True)

        ax.pcolormesh(
            dem_lon_2d, dem_lat_2d, dem_data,
            cmap      = cmap_elev,
            norm      = norm_topo,
            alpha     = 1.0,
            transform = ccrs.PlateCarree(),
        )
        ax.contour(
            dem_lon_2d, dem_lat_2d, dem_data,
            levels    = [0.0],
            colors    = 'black',
            linewidths= 0.5,
            transform = ccrs.PlateCarree(),
            zorder    = 15,
        )
        ax.contour(
            dem_lon_2d, dem_lat_2d, dem_data,
            levels    = [400.0],
            colors    = 'black',
            linewidths= 0.3,
            transform = ccrs.PlateCarree(),
            zorder    = 15,
        )
        print('Terrain shading loaded successfully')
    else:
        raise ValueError('GEBCO DEM returned None')

except Exception as e:
    print(f'Terrain shading failed: {e}')
    ax.add_feature(cfeature.OCEAN, facecolor='lightblue', alpha=0.3, zorder=1)
    ax.add_feature(cfeature.LAND,  facecolor='#E8E8E8',   alpha=0.3, zorder=2)

ax.add_feature(cfeature.BORDERS, linewidth=0.5, alpha=0.5, zorder=3)

if (PlotType == 'scatter'):
    # ── plot birth locations coloured by time of day ──────────────────────────────
    sc = ax.scatter(
        birth_lons, birth_lats,
        c          = birth_mins,
        cmap       = plt.get_cmap('nipy_spectral'),
        vmin       = 0,
        vmax       = 1440,
        s          = 6,             # ← smaller dots
        alpha      = 0.8,
        transform  = ccrs.PlateCarree(),
        zorder     = 25,
        edgecolors = 'none',
    )

elif (PlotType == 'density'):
    # ── kernel density estimate of birth locations ────────────────────────────────
    xy          = np.vstack([birth_lons, birth_lats])
    kde         = gaussian_kde(xy, bw_method=0.1)
    
    lon_grid    = np.linspace(lon_min, lon_max, 300)
    lat_grid    = np.linspace(lat_min, lat_max, 300)
    lon_mesh, lat_mesh = np.meshgrid(lon_grid, lat_grid)
    grid_coords = np.vstack([lon_mesh.ravel(), lat_mesh.ravel()])
    
    kde_values  = kde(grid_coords).reshape(lon_mesh.shape)
    
    # ── convert KDE density to births per 100 km² per day ────────────────────────
    # KDE output units: probability density per degree²
    # 1 degree latitude  ≈ 111.32 km
    # 1 degree longitude ≈ 111.32 * cos(mean_lat) km
    mean_lat      = np.mean(birth_lats)
    km_per_deg_lat = 111.32
    km_per_deg_lon = 111.32 * np.cos(np.radians(mean_lat))
    km2_per_deg2   = km_per_deg_lat * km_per_deg_lon   # km² per degree²
    
    # Multiply density (per deg²) by km² per deg² to get per km²
    # Multiply by n_births to get births per km²
    # Multiply by 100 to get births per 100 km²
    n_births       = len(birth_lats)
    kde_births_per_100km2 = (kde_values / km2_per_deg2) * n_births * 100.0
    
    # ── shaded KDE fill ───────────────────────────────────────────────────────────
    levels_fill = np.arange(0, 62.5, 2.5)
    
    # Full range plot — used only to drive the colourbar, fully transparent
    kde_fill = ax.contourf(
        lon_mesh, lat_mesh, kde_births_per_100km2,
        levels    = levels_fill,
        cmap      = 'jet',
        alpha     = 0.0,             # invisible — just for colourbar
        transform = ccrs.PlateCarree(),
        zorder    = 24,
        extend    = 'max',
    )
    
    # Visible plot — only shows values >= 5
    kde_fill_visible = ax.contourf(
        lon_mesh, lat_mesh, kde_births_per_100km2,
        levels    = np.arange(2.5, 62.5, 2.5),   # starts at 5, skips the 0-5 bin
        cmap      = 'jet',
        alpha     = 0.5,
        transform = ccrs.PlateCarree(),
        zorder    = 24,
        extend    = 'max',
    )
    
    # ── colourbar ─────────────────────────────────────────────────────────────────
    cax = fig.add_axes([
        ax.get_position().x1 + 0.025,
        ax.get_position().y0 - 0.025,
        0.02,
        ax.get_position().height
    ])
    
    # Use a ScalarMappable with the full 0-100 range so the colourbar
    # shows the complete scale even though the 0-5 bin is not plotted
    sm = plt.cm.ScalarMappable(
        cmap = 'jet',
        norm = plt.Normalize(vmin=0, vmax=60)
    )
    sm.set_array([])
    
    cbar = plt.colorbar(sm, cax=cax, orientation='vertical', extend='max')
    cbar.set_label('Feature Births per 100 km² per day', fontsize=9)
    cbar.set_ticks(np.arange(0, 62.5, 2.5))
    ticks = np.arange(0, 62.5, 2.5)
    
    cbar.set_ticklabels([
        f'{int(v)}' if i % 2 == 0 else ''
        for i, v in enumerate(ticks)
    ])

    cbar.ax.tick_params(labelsize=8)

else:
    print("PlotType must be exactly 'scatter' or 'density' as a string")

# ── gridlines ─────────────────────────────────────────────────────────────────
gl_minor = ax.gridlines(draw_labels=False, alpha=0.8, zorder=11, linewidth=0.2)
gl_minor.xlocator = mticker.MultipleLocator(0.1)
gl_minor.ylocator = mticker.MultipleLocator(0.1)

gl_mid = ax.gridlines(draw_labels=True, alpha=0.8, zorder=12, linewidth=0.3)
gl_mid.xlocator = mticker.MultipleLocator(0.5)
gl_mid.ylocator = mticker.MultipleLocator(0.5)

gl_major = ax.gridlines(draw_labels=True, alpha=0.8, zorder=13, linewidth=0.6)
gl_major.xlocator = mticker.MultipleLocator(1.0)
gl_major.ylocator = mticker.MultipleLocator(1.0)

gl_mid.xformatter = LONGITUDE_FORMATTER
gl_mid.yformatter = LATITUDE_FORMATTER
gl_major.xformatter = LONGITUDE_FORMATTER
gl_major.yformatter = LATITUDE_FORMATTER

gl_mid.top_labels   = False
gl_mid.right_labels = False
gl_major.top_labels   = False
gl_major.right_labels = False

ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')

if (PlotType == 'scatter'):
    ax.set_title(
        f'Feature (dBZ > {FeatureDBZmin}) "Birth" Locations for {RadarSiteName} Radar\n'
        f'at {Altitude*0.001} km Altitude on {RadarFileDatePrint} UTC\n'
        f'({len(birth_lats)} Features)',
        fontsize=12
    )

    SaveFile   = (RadarIDno + '_' + RadarFileDate + '_' +
              str(int(FeatureDBZmin)) + 'DBZmin_BirthLocations_' + str(Altitude) + 'm.png')

        # ── time of day colourbar ─────────────────────────────────────────────────────
    cax = fig.add_axes([
        ax.get_position().x1 + 0.025,    # just right of the map
        ax.get_position().y0 + 0.025,            # bottom aligned with map
        0.02,                            # width
        ax.get_position().height         # full height of map
    ])
    
    cbar = plt.colorbar(sc, cax=cax, orientation='vertical')
    cbar.set_label('Birth Time (UTC)')
    
    hourly_ticks = np.arange(0, 1441, 60)
    cbar.set_ticks(hourly_ticks)
    tick_labels = [
        f'{int(m // 60):02d}:00' if m % 180 == 0 else ''
        for m in hourly_ticks
    ]
    cbar.set_ticklabels(tick_labels)
    cbar.ax.tick_params(which='major', length=4, width=0.8)

elif (PlotType == 'density'):
    ax.set_title(
        f'Feature (dBZ > {FeatureDBZmin}) "Birth" Locations Density for {RadarSiteName} Radar\n'
        f'at {Altitude*0.001} km Altitude on {RadarFileDatePrint} UTC\n'
        f'({len(birth_lats)} Features)',
        fontsize=12
    )

    SaveFile   = (RadarIDno + '_' + RadarFileDate + '_' +
              str(int(FeatureDBZmin)) + 'DBZmin_BirthLocationDensity_' + str(Altitude) + 'm.png')
else:
    print("PlotType must be exactly 'scatter' or 'density' as a string")


plt.tight_layout()

# ── save ──────────────────────────────────────────────────────────────────────
SaveFolder = ('/scratch/v46/sg3241/tmp/pngImages/FeatureTracks/' +
              RadarIDno + '/' + RadarFileDate + '/')
SavePath   = SaveFolder + SaveFile

if not Path(SaveFolder).exists():
    Path(SaveFolder).mkdir(parents=True, exist_ok=True)

plt.savefig(SavePath, bbox_inches='tight', facecolor='w', dpi=300)
plt.close()
print(f"Saved to {SavePath}")


In [ ]:
# CHAD PLOT
# GREAT BIG AND GRAND
# FEATURE LOCATIONS (BIRTH DEATH AND ALL) AND KERENEL DENSITY

def PlotFeatureLocations(
    RadarIDno,
    RadarSiteName,
    QualityControlOption,
    DateRange,
    TimeRange,
    DotsIncluded,
    MinDuration,
):
    """
    Plots feature locations (birth, death, or all frames) across a date and
    time range, loaded from daily PyFLEXTRKR NetCDF files.

    Produces two plots:
        1. Scatter plot  — small red dots on terrain background
        2. Density plot  — kernel density estimate on terrain background

    Parameters
    ----------
    RadarIDno            : str   - Radar ID number string (e.g. '22')
    RadarSiteName        : str   - Radar site name for plot title (e.g. 'Mackay')
    QualityControlOption : int   - QC option number
    DateRange            : str   - 'YYYYMMDD-YYYYMMDD' inclusive date range
    TimeRange            : str   - 'HH:MM-HH:MM' UTC time-of-day window
    DotsIncluded         : str   - 'Birth', 'Death', or 'All'
    MinDuration          : float - Minimum feature duration in minutes.
                                   Feature must last at least ceil(MinDuration/5)+1 frames.
    """

    import math
    import calendar
    import pandas as pd
    from scipy.stats           import gaussian_kde
    from pathlib               import Path
    from matplotlib.colors     import ListedColormap, BoundaryNorm
    from matplotlib.lines      import Line2D
    import matplotlib.ticker   as mticker
    from cartopy.mpl.gridliner import LONGITUDE_FORMATTER, LATITUDE_FORMATTER
    import cartopy.crs         as ccrs
    import cartopy.feature     as cfeature

    # ── Input validation ───────────────────────────────────────────────────────

    if DotsIncluded not in ('Birth', 'Death', 'All'):
        raise ValueError("DotsIncluded must be 'Birth', 'Death', or 'All'.")

    # --- Parse date range ---
    date_start_str = DateRange[:8]
    date_end_str   = DateRange[9:]
    date_start     = pd.Timestamp(f'{date_start_str[:4]}-{date_start_str[4:6]}-{date_start_str[6:8]}')
    date_end       = pd.Timestamp(f'{date_end_str[:4]}-{date_end_str[4:6]}-{date_end_str[6:8]}')
    all_dates      = pd.date_range(date_start, date_end, freq='D')

    # --- Parse time range ---
    time_start_str, time_end_str = TimeRange.split('-')
    t_start_h, t_start_m = int(time_start_str.split(':')[0]), int(time_start_str.split(':')[1])
    t_end_h,   t_end_m   = int(time_end_str.split(':')[0]),   int(time_end_str.split(':')[1])

    t_start_mins = t_start_h * 60 + t_start_m
    t_end_mins   = t_end_h   * 60 + t_end_m

    # --- Reject across-midnight ranges ---
    if t_end_mins <= t_start_mins:
        raise ValueError(
            f"TimeRange '{TimeRange}' appears to span midnight or has equal start/end. "
            "Across-midnight ranges are not currently supported. "
            "Please provide a time range within a single calendar day (e.g. '06:00-18:00')."
        )

    # --- Minimum frame count from MinDuration ---
    MinFrames = math.ceil(MinDuration / 5) + 1

    print(f"Duration filter : >= {MinDuration} min  →  >= {MinFrames} frames")
    print(f"Date range      : {date_start.date()} to {date_end.date()} ({len(all_dates)} days)")
    print(f"Time window     : {TimeRange} UTC")
    print(f"Mode            : {DotsIncluded}")

    # ── Collect lat/lon points across all days ─────────────────────────────────

    all_lats = []
    all_lons = []

    for date in all_dates:

        FileDateStr = date.strftime('%Y%m%d')
        print(f'Working on {FileDateStr}')

        FeatureStoragePath = (
            f'/scratch/v46/sg3241/tmp/NetCDFs/PyFLEXTRKR/Stats/'
            f'{RadarIDno}/{FileDateStr}/QC{QualityControlOption}/V5/'
            f'trackstats_{FileDateStr}.000000_{FileDateStr}.235500.nc'
        )

        try:
            FeatureXR = xr.open_dataset(FeatureStoragePath)
        except FileNotFoundError:
            print(f"  File not found for {FileDateStr}, skipping.")
            continue

        FeatureXR = AddPropagationVars(FeatureXR)
        FeatureXR = AddFrameTimeVars(FeatureXR)

        base_time      = FeatureXR['base_time'].values        # (tracks, times)
        meanlat        = FeatureXR['meanlat'].values          # (tracks, times)
        meanlon        = FeatureXR['meanlon'].values          # (tracks, times)
        track_duration = FeatureXR['track_duration'].values   # (tracks,)

        n_tracks = base_time.shape[0]

        for track_i in range(n_tracks):

            # --- Duration filter ---
            if track_duration[track_i] < MinFrames:
                continue

            track_times = base_time[track_i, :]   # (times,)
            track_lats  = meanlat [track_i, :]
            track_lons  = meanlon [track_i, :]

            # --- Find valid (non-NaT, non-NaN) frames ---
            valid_mask = (
                ~np.isnat(track_times.astype('datetime64[ns]')) &
                np.isfinite(track_lats) &
                np.isfinite(track_lons)
            )

            if not np.any(valid_mask):
                continue

            valid_times = track_times[valid_mask]
            valid_lats  = track_lats [valid_mask]
            valid_lons  = track_lons [valid_mask]

            # --- Convert valid times to time-of-day in minutes ---
            valid_pd    = pd.to_datetime(valid_times)
            tod_mins    = np.array([
                t.hour * 60 + t.minute + t.second / 60.0
                for t in valid_pd
            ])

            # --- Select points based on DotsIncluded ---
            if DotsIncluded == 'Birth':
                # Only the first valid frame — check if it falls in time window
                candidate_tod  = tod_mins[0]
                candidate_lats = [valid_lats[0]]
                candidate_lons = [valid_lons[0]]
                candidate_tods = [candidate_tod]

            elif DotsIncluded == 'Death':
                # Only the last valid frame — check if it falls in time window
                candidate_tod  = tod_mins[-1]
                candidate_lats = [valid_lats[-1]]
                candidate_lons = [valid_lons[-1]]
                candidate_tods = [candidate_tod]

            else:  # 'All'
                candidate_lats = valid_lats
                candidate_lons = valid_lons
                candidate_tods = tod_mins

            # --- Apply time-of-day filter ---
            for lat, lon, tod in zip(candidate_lats, candidate_lons, candidate_tods):
                if t_start_mins <= tod <= t_end_mins:
                    all_lats.append(lat)
                    all_lons.append(lon)

        FeatureXR.close()

    all_lats = np.array(all_lats)
    all_lons = np.array(all_lons)

    print(f"\n{len(all_lats)} points collected for plotting.")

    if len(all_lats) == 0:
        print("No valid points found — nothing to plot.")
        return

    # ── Shared map extent ──────────────────────────────────────────────────────

    lon_min = float(np.nanmin(all_lons))
    lon_max = float(np.nanmax(all_lons))
    lat_min = float(np.nanmin(all_lats))
    lat_max = float(np.nanmax(all_lats))

    # ── Shared title suffix ────────────────────────────────────────────────────

    title_suffix = (
        f'{RadarSiteName} Radar  |  {DotsIncluded} Locations\n'
        f'{date_start.strftime("%Y-%m-%d")} to {date_end.strftime("%Y-%m-%d")}  |  '
        f'{TimeRange} UTC  |  Min Duration: {MinDuration} min  |  '
        f'({len(all_lats)} points)'
    )

    # ── Save folder ────────────────────────────────────────────────────────────

    SaveFolder = (
        f'/scratch/v46/sg3241/tmp/pngImages/FeatureLocations/'
        f'{RadarIDno}/{DateRange}/'
    )
    if not Path(SaveFolder).exists():
        Path(SaveFolder).mkdir(parents=True, exist_ok=True)

    SaveBase = (
        f'{RadarIDno}_{DateRange}_{TimeRange.replace(":", "").replace("-", "_")}_'
        f'{DotsIncluded}_{int(MinDuration)}minMin'
    )

    # ── Helper: build base map ─────────────────────────────────────────────────

    def build_basemap():

        fig, ax = plt.subplots(
            figsize     = (8, 6),
            subplot_kw  = {'projection': ccrs.PlateCarree()},
            facecolor   = 'white'
        )

        # --- Terrain background ---
        try:
            gebco_path = '/home/563/sg3241/QueenslandElevationGEBCO.nc'
            dem_da     = fetch_gebco_local(gebco_path, lon_min, lon_max, lat_min, lat_max)

            if dem_da is None:
                raise ValueError('GEBCO DEM returned None')

            dem_lon  = dem_da.lon.values
            dem_lat  = dem_da.lat.values
            dem_data = dem_da.values

            if dem_lon.ndim == 1 and dem_lat.ndim == 1:
                dem_lon_2d, dem_lat_2d = np.meshgrid(dem_lon, dem_lat)
            else:
                dem_lon_2d, dem_lat_2d = dem_lon, dem_lat

            if not np.any(np.isfinite(dem_data)):
                raise ValueError('DEM has no finite values in this domain')

            colours_topo = [
                '#dde4e8', '#c4dec2', '#e4edc9', '#f3f0cf',
                '#e9d7bd', '#ddc4aa', '#cfb194', '#b58f6e',
            ]
            bounds_topo = [-1000.0, 0.0, 200.0, 400.0, 600.0, 800.0, 1000.0, 1200.0, 5000.0]
            cmap_elev   = ListedColormap(colours_topo)
            norm_topo   = BoundaryNorm(bounds_topo, len(colours_topo), clip=True)

            ax.pcolormesh(
                dem_lon_2d, dem_lat_2d, dem_data,
                cmap=cmap_elev, norm=norm_topo,
                alpha=1.0, transform=ccrs.PlateCarree(),
            )
            ax.contour(
                dem_lon_2d, dem_lat_2d, dem_data,
                levels=[0.0], colors='black', linewidths=0.5,
                transform=ccrs.PlateCarree(), zorder=15,
            )
            ax.contour(
                dem_lon_2d, dem_lat_2d, dem_data,
                levels=[400.0], colors='black', linewidths=0.3,
                transform=ccrs.PlateCarree(), zorder=15,
            )
            print('  Terrain shading loaded successfully.')

        except Exception as e:
            print(f'  Terrain shading failed: {e}')
            ax.add_feature(cfeature.OCEAN, facecolor='lightblue', alpha=0.3, zorder=1)
            ax.add_feature(cfeature.LAND,  facecolor='#E8E8E8',   alpha=0.3, zorder=2)

        ax.add_feature(cfeature.BORDERS, linewidth=0.5, alpha=0.5, zorder=3)

        # --- Gridlines ---
        gl_minor = ax.gridlines(draw_labels=False, alpha=0.8, zorder=11, linewidth=0.2)
        gl_minor.xlocator = mticker.MultipleLocator(0.1)
        gl_minor.ylocator = mticker.MultipleLocator(0.1)

        gl_mid = ax.gridlines(draw_labels=True, alpha=0.8, zorder=12, linewidth=0.3)
        gl_mid.xlocator = mticker.MultipleLocator(0.5)
        gl_mid.ylocator = mticker.MultipleLocator(0.5)

        gl_major = ax.gridlines(draw_labels=True, alpha=0.8, zorder=13, linewidth=0.6)
        gl_major.xlocator = mticker.MultipleLocator(1.0)
        gl_major.ylocator = mticker.MultipleLocator(1.0)

        for gl in (gl_mid, gl_major):
            gl.xformatter    = LONGITUDE_FORMATTER
            gl.yformatter    = LATITUDE_FORMATTER
            gl.top_labels    = False
            gl.right_labels  = False

        ax.set_xlabel('Longitude')
        ax.set_ylabel('Latitude')

        return fig, ax

    # ══════════════════════════════════════════════════════════════════════════
    # PLOT 1 — Scatter (red dots)
    # ══════════════════════════════════════════════════════════════════════════

    print('\nGenerating scatter plot...')
    fig1, ax1 = build_basemap()

    ax1.scatter(
        all_lons, all_lats,
        s          = 6,
        color      = 'red',
        alpha      = 0.6,
        transform  = ccrs.PlateCarree(),
        zorder     = 25,
        edgecolors = 'none',
    )

    ax1.set_title(f'Feature {title_suffix}', fontsize=10)

    plt.tight_layout()
    SavePath1 = SaveFolder + SaveBase + '_Scatter.png'
    # plt.savefig(SavePath1, bbox_inches='tight', facecolor='w', dpi=300)
    # plt.close()
    print(f'  Scatter plot saved to {SavePath1}')

    # ══════════════════════════════════════════════════════════════════════════
    # PLOT 2 — Kernel Density
    # ══════════════════════════════════════════════════════════════════════════

    print('\nGenerating density plot...')
    fig2, ax2 = build_basemap()

    xy         = np.vstack([all_lons, all_lats])
    kde        = gaussian_kde(xy, bw_method=0.1)

    lon_grid   = np.linspace(lon_min, lon_max, 300)
    lat_grid   = np.linspace(lat_min, lat_max, 300)
    lon_mesh, lat_mesh = np.meshgrid(lon_grid, lat_grid)
    grid_coords = np.vstack([lon_mesh.ravel(), lat_mesh.ravel()])

    kde_values  = kde(grid_coords).reshape(lon_mesh.shape)

    # --- Convert to points per 100 km² ---
    mean_lat        = np.mean(all_lats)
    km_per_deg_lat  = 111.32
    km_per_deg_lon  = 111.32 * np.cos(np.radians(mean_lat))
    km2_per_deg2    = km_per_deg_lat * km_per_deg_lon
    n_points        = len(all_lats)
    kde_per_100km2  = (kde_values / km2_per_deg2) * n_points * 100.0

    # --- Dynamic colour scale ---
    kde_max        = float(np.nanmax(kde_per_100km2))
    level_step     = kde_max / 25.0          # 25 intervals across the range
    levels_fill    = np.arange(0, kde_max + level_step, level_step)
    levels_visible = levels_fill[1:]         # skip the 0 bin (invisible)

    # Invisible contourf — drives the colourbar over the full 0→max range
    kde_fill = ax2.contourf(
        lon_mesh, lat_mesh, kde_per_100km2,
        levels    = levels_fill,
        cmap      = 'jet',
        alpha     = 0.0,
        transform = ccrs.PlateCarree(),
        zorder    = 24,
        extend    = 'max',
    )

    # Visible contourf — starts from the first non-zero bin
    ax2.contourf(
        lon_mesh, lat_mesh, kde_per_100km2,
        levels    = levels_visible,
        cmap      = 'jet',
        alpha     = 0.25,
        transform = ccrs.PlateCarree(),
        zorder    = 24,
        extend    = 'max',
    )

    # --- Colourbar ---
    cax = fig2.add_axes([
        ax2.get_position().x1 + 0.025,
        ax2.get_position().y0 - 0.025,
        0.02,
        ax2.get_position().height,
    ])

    sm = plt.cm.ScalarMappable(
        cmap = 'jet',
        norm = plt.Normalize(vmin=0, vmax=kde_max)
    )
    sm.set_array([])

    cbar = plt.colorbar(sm, cax=cax, orientation='vertical', extend='max')
    cbar.set_label('Feature Locations per 100 km²', fontsize=9)

    # Show ~10 labelled ticks evenly spaced
    tick_vals = np.linspace(0, kde_max, 11)
    cbar.set_ticks(tick_vals)
    cbar.set_ticklabels([f'{v:.1f}' for v in tick_vals])
    cbar.ax.tick_params(labelsize=8)

    ax2.set_title(f'Feature Density {title_suffix}', fontsize=10)

    plt.tight_layout()
    plt.draw()

    # Realign colourbar after tight_layout shifts axes
    cax.set_position([
        ax2.get_position().x1 + 0.025,
        ax2.get_position().y0 - 0.025,
        0.02,
        ax2.get_position().height,
    ])

    SavePath2 = SaveFolder + SaveBase + '_Density.png'
    # plt.savefig(SavePath2, bbox_inches='tight', facecolor='w', dpi=300)
    # plt.close()
    print(f'  Density plot saved to {SavePath2}')


In [ ]:
PlotFeatureLocations(
    RadarIDno            = '22',
    RadarSiteName        = 'Mackay',
    QualityControlOption = 2,
    DateRange            = '20240201-20240228',
    TimeRange            = '18:00-22:00',
    DotsIncluded         = 'Birth', # 'Birth', 'Death', or 'All'
    MinDuration          = 22, # only include features that last longer than this in min
)
